# Register WNV Assistant Model

Logs the agent as an MLflow model and registers it to Unity Catalog.

In [ ]:
# Get the repo filesystem path (Repos are mounted under /Repos/)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Extract repo path from notebook location
repo_fs_path = '/Repos/' + notebook_path.split('/Repos/')[1].rsplit('/', 2)[0] if '/Repos/' in notebook_path else '/Repos/eliao@bpcs.com/enterprise_databricks_ai_assistance'
print(f'Repo filesystem path: {repo_fs_path}')

# Install from repo
%pip install -e {repo_fs_path} databricks-sql-connector mlflow
dbutils.library.restartPython()

In [ ]:
import mlflow

# Register models in the Unity Catalog Model Registry, not the legacy workspace registry.
mlflow.set_registry_uri("databricks-uc")
import sys
sys.path.insert(0, repo_fs_path)

from wnv_assistant.serving import log_model

# Log the model
model_uri = log_model("/tmp/wnv-model")
print(f"Model logged: {model_uri}")

In [ ]:
# Register to Unity Catalog
model_version = mlflow.register_model(
    model_uri=model_uri,
    name="eliao.wnv_demo.wnv_assistant",
)
print(f"Registered: {model_version}")

In [ ]:
# Test the logged model
import mlflow.pyfunc

model = mlflow.pyfunc.load_model(model_uri)
result = model.predict({"question": "Show top 5 counties by mosquito activity in 2022"})
print(result)